# 2. Native cells, area weighting and NoData

Native counts are allocated by `cell_count × spherical_overlap_area / spherical_cell_area`. This assumes uniform population within each cell; it cannot recover unknown settlement patterns. Counts are not densities. The calculation preserves different grid offsets and null masks.


In [ ]:
import json
from pathlib import Path
from population_tools_real import load_population, compare, validate_layer
data = load_population()
oracle = json.loads(Path('independent_oracle.json').read_text(encoding='utf-8'))
for layer in data['layers']:
    validate_layer(layer)


## Independently inspect a native pixel

A longitude half-cell has half the spherical area. This checks a real GHS-POP pixel. Each layer's own native extent also reproduces its independently calculated source sum.


In [ ]:
from population_tools_real import iter_cells, layer_bounds
ghs = data['layers'][3]
cell = next(c for c in iter_cells(ghs) if c['population'] > 0)
w, s, e, n = cell['bounds']
half = compare([ghs], [w, s, (w+e)/2, n])[0]
assert abs(half['population'] - cell['population']/2) < 1e-6
for layer in data['layers']:
    result = compare([layer], layer_bounds(layer))[0]
    expected = oracle['sources'][layer['metadata']['id']]
    assert abs(result['population'] - expected['native_sum']) < .001
    print(layer['metadata']['id'], result['population'], result['coverage'])


## Zero remains distinct from missing

GHS-POP declares no NoData; valid zeros include water. GRID3 NaN denotes mapped unsettled inside Nigeria and outside coverage outside the boundary. It is preserved as null, not zero. Numeric-cell coverage is a data-support measure, not a confidence or accuracy score.


In [ ]:
zero_cell = next(c for c in iter_cells(ghs) if c['population'] == 0)
zero = compare([ghs], zero_cell['bounds'])[0]
assert zero['population'] == 0 and abs(zero['coverage']-1) < 1e-10
missing = compare(data['layers'], [0, 0, 1, 1])[0]
assert missing['population'] is None and missing['coverage'] == 0
assert any(v is None for v in data['layers'][1]['grid']['values'])
print('Covered zero:', zero['population'], 'Missing:', missing['population'])
print('REAL_COVERAGE_BROWSER_OK')


## Sources and software

The complete citation, release, native geometry, source asset, NoData semantics, license, retrieval date (2026-11-06) and processing receipt travel in every layer's metadata. These are native-pixel Greater Lagos excerpts, spatially subset and losslessly serialized without resampling; no producer endorsement. All four products are CC BY 4.0. WorldPop: Bondarenko et al. (2025), https://doi.org/10.5258/SOTON/WP00839 (Global2 R2025A v1 alpha); GRID3: Nnanatu et al. (2025), https://doi.org/10.5258/SOTON/WP00782; LandScan: Rose et al. (2021), https://doi.org/10.48690/1523378; GHS-POP: Schiavina et al. (2023), https://doi.org/10.2905/2FF68A52-5B5B-4A22-8F40-C41DA8332CFE and Pesaresi et al. (2024), https://doi.org/10.1080/17538947.2024.2390454. Credit WorldPop / University of Southampton; GRID3 Inc. and CIESIN / Columbia Climate School; Oak Ridge National Laboratory; European Commission/JRC, (c) European Union, 1995-2026. Gates Foundation WorldPop INV-045237 and GRID3 INV-044979. Preserve all producer notices when sharing.

LandScan pixels came from a pinned Hugging Face mirror, not an ORNL-hosted TIFF. Independent ORNL statistics matched one cell and the complete prepared window; they do not prove full mirror bit identity. GHS-POP uses the official JRC source, not a quantized third-party mirror. GPW is not bundled; current Harvard Dataverse access uses a guestbook. GRID3 and WorldPop file servers currently lack browser CORS/range support; download yourself and use the viewer's bounded local TIFF import for other areas.

Inspired by NASA Earthdata, SEDAC/CIESIN and POPGRID Viewer (https://www.earthdata.nasa.gov/news/feature-articles/data-tool-focus-popgrid-viewer). Fitness for use: Leyk et al. (2019), https://doi.org/10.5194/essd-11-1385-2019. Execution: Project Jupyter / JupyterLite, Pyodide, Jupyter Book / MyST. The viewer uses Leaflet 1.9.4 and GeoTIFF.js 3.0.5 with their notices. RustGIS, GeoLibre, JupyterGIS and FieldWork integration remain future work.
